In [6]:
from JAX_QNN import quantum_neural_network
from QNN_jax import initialize_parameters
import pennylane as qml
import jax
from jax import numpy as jnp, random
from flax import nnx


In [ ]:
dev = qml.device("default.qubit", wires=4)
qnode = qml.QNode(func=quantum_neural_network, device=dev,interface='jax')
qnode(x=[1,0,1,0],params=initialize_parameters(2,4),n_layers=2,n_qubits=4)

In [8]:
# 定义量子设备和 QNode
dev = qml.device("default.qubit", wires=4)
qnode = qml.QNode(func=quantum_neural_network, device=dev, interface='jax')

# 定义输入数据
x = jnp.array([1, 0, 1, 0])  # 输入数据
params = initialize_parameters(2, 4)  # 初始化参数
n_layers = 2
n_qubits = 4

# 计算梯度（对参数求导）
grad_fn = jax.grad(qnode)  # 默认对第一个参数（x）求导
grad_x = grad_fn(x, params, n_layers, n_qubits)
print("Gradient w.r.t. x:", grad_x)

# 计算对参数的梯度
grad_params_fn = jax.grad(qnode, argnums=1)  # 对第二个参数（params）求导
grad_params = grad_params_fn(x, params, n_layers, n_qubits)
print("Gradient w.r.t. params:", grad_params)


TypeError: grad requires real- or complex-valued inputs (input dtype that is a sub-dtype of np.inexact), but got int32. If you want to use Boolean- or integer-valued inputs, use vjp or set allow_int to True.

In [ ]:
import os
import netket as nk
from scipy.sparse.linalg import eigsh
from netket.operator.spin import sigmax, sigmaz

N = 6
hi = nk.hilbert.Spin(s=1 / 2, N=N)
os.environ["JAX_PLATFORM_NAME"] = "cpu"
hi.random_state(jax.random.key(0), 3)
Gamma = -1
H = sum([Gamma * sigmax(hi, i) for i in range(N)])
V = -1
H += sum([V * sigmaz(hi, i) @ sigmaz(hi, (i + 1) % N) for i in range(N)])
sp_h = H.to_sparse()
eig_vals, eig_vecs = eigsh(sp_h, k=2, which="SA")
print("eigenvalues with scipy sparse:", eig_vals)
E_gs = eig_vals[0]

# Create the local sampler on the hilbert space
sampler = nk.sampler.MetropolisLocal(hi)

In [ ]:
class FFN(nnx.Module):

    def __init__(self, N: int, alpha: int = 1, *, rngs: nnx.Rngs):
        """
        Construct a Feed-Forward Neural Network with a single hidden layer.

        Args:
            N: The number of input nodes (number of spins in the chain).
            alpha: The density of the hidden layer. The hidden layer will have
                N*alpha nodes.
            rngs: The random number generator seed.
        """
        self.alpha = alpha

        # We define a linear (or dense) layer with `alpha` times the number of input nodes
        # as output nodes.
        # We must pass forward the rngs object to the dense layer.
        self.linear = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs)

    def __call__(self, x: jax.Array):
        print(f'input X={x}')
        # we apply the linear layer to the input
        y = self.linear(x)

        # the non-linearity is a simple ReLu
        y = nnx.relu(y)

        # sum the output
        return jnp.sum(y, axis=-1)


model = FFN(N=N, alpha=1, rngs=nnx.Rngs(2))

vstate = nk.vqs.MCState(sampler, model, n_samples=1008)

In [ ]:
optimizer = nk.optimizer.Sgd(learning_rate=0.1)

# Notice the use, again of Stochastic Reconfiguration, which considerably improves the optimisation
gs = nk.driver.VMC(
    H,
    optimizer,
    variational_state=vstate,
    preconditioner=nk.optimizer.SR(diag_shift=0.1),
)

log = nk.logging.RuntimeLog()
gs.run(n_iter=300, out=log)

ffn_energy = vstate.expect(H)
error = abs((ffn_energy.mean - eig_vals[0]) / eig_vals[0])
print("Optimized energy and relative error: ", ffn_energy, error)

In [ ]:
from matplotlib import pyplot as plt
plt.errorbar(
    log.data["Energy"].iters[50:],
    log.data["Energy"].Mean[50:],
    yerr=log.data["Energy"].Sigma[50:],
    label="SymmModel",
)

plt.axhline(
    y=eig_vals[0],
    xmin=0,
    xmax=log.data["Energy"].iters[-1],
    linewidth=2,
    color="k",
    label="Exact",
)
plt.xlabel("Iterations")
plt.ylabel("Energy")
plt.legend(frameon=False)

In [ ]:
class QuantumAnsatz(nnx.Module):
    def __init__(self, n_qubits: int, n_layers: int, rngs: nnx.Rngs):
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.param_shape = (self.n_layers, 2 * n_qubits)
        # 关键修复：为 JAX 随机数生成器指定 dtype
        self.qnn_params = nnx.Param(
            jax.random.uniform(
                rngs.params(), 
                shape=self.param_shape, 
                minval=-jnp.pi, 
                maxval=jnp.pi,
                dtype=jnp.float32  # <--- 明确指定为 float32
            )
        )
        
        # 定义量子设备和 QNode
        self.device = qml.device("default.qubit", wires=self.n_qubits)
        self.qnn_node = qml.QNode(func=quantum_neural_network, device=self.device, interface='jax')

        # 关键修复：定义一个辅助函数来包装 QNode 并转换输出
        def _qnode_wrapper(x):
            list_output = self.qnn_node(x, self.n_qubits, self.qnn_params.value, self.n_layers)
            return jnp.stack(list_output)

        # 在 __init__ 中对辅助函数进行向量化
        # 辅助函数只有一个参数，所以 in_axes=(0,)
        self.vectorized_qnode = jax.vmap(_qnode_wrapper, in_axes=(0,))

        # 经典层
        self.after_qnn = nnx.Linear(
            in_features=self.n_qubits, out_features=1, use_bias=False, rngs=rngs
        )

    def __call__(self, configuration: jnp.ndarray):
        # 1. 量子电路处理批数据
        qnn_output = self.vectorized_qnode(configuration)
        # 现在 qnn_output 的形状是 (batch_size, n_qubits)，例如 (16, 3)
        # 2. 经典层处理
        after_qnn_output = self.after_qnn(qnn_output)
        # 3. 调整输出形状以匹配 NetKet 的期望
        return jnp.squeeze(after_qnn_output, axis=-1)



In [ ]:
qnn_model = QuantumAnsatz(n_qubits=6,n_layers=2,rngs=nnx.Rngs(params=0))
qnn_vstate = nk.vqs.MCState(sampler, qnn_model, n_samples=1008)

In [ ]:
qnn_optimizer = nk.optimizer.Sgd(learning_rate=0.1)

# Notice the use, again of Stochastic Reconfiguration, which considerably improves the optimisation
qnn_gs = nk.driver.VMC(
    H,
    qnn_optimizer,
    variational_state=qnn_vstate,
    preconditioner=nk.optimizer.SR(diag_shift=0.1),
)

In [ ]:
qnn_log = nk.logging.RuntimeLog()
qnn_gs.run(n_iter=300, out=qnn_log)

qnn_ffn_energy = qnn_vstate.expect(H)
qnn_error = abs((qnn_ffn_energy.mean - eig_vals[0]) / eig_vals[0])
print("Optimized energy and relative error: ", qnn_ffn_energy, qnn_error)

In [ ]:
from matplotlib import pyplot as plt
plt.errorbar(
    qnn_log.data["Energy"].iters[50:],
    qnn_log.data["Energy"].Mean[50:],
    yerr=qnn_log.data["Energy"].Sigma[50:],
    label="SymmModel",
)

plt.axhline(
    y=eig_vals[0],
    xmin=0,
    xmax=qnn_log.data["Energy"].iters[-1],
    linewidth=2,
    color="k",
    label="Exact",
)
plt.xlabel("Iterations")
plt.ylabel("Energy")
plt.legend(frameon=False)

In [ ]:
qnn_log.data["Energy"]